#  Byte Pair Encoding (BPE) algorithm

Our primary focus will be on understanding and implementing the Byte Pair Encoding (BPE) algorithm, a popular method for subword tokenization.

In [1]:
import string


For the tasks, we will be working with the renowned play "Hamlet" by William Shakespeare. "Hamlet" is a masterpiece of English literature, celebrated for its profound themes, intricate characters, and rich linguistic tapestry.

The following cell loads the "Hamlet" text data.

In [2]:
def read_data(file_path):
    # Open the file in read mode ('r')
    with open(file_path, 'r') as file:
        # Read the entire file content
        return file.read()
        
text = read_data('data/hamlet.txt')


Text is prepared by removing punctuation and converting to lower case:

In [3]:
def clean_and_normalize_text(text):
    """
    Clean and normalize the input text by converting it to lowercase
    and removing punctuation.

    Parameters:
    - text (str): The input text to be cleaned.

    Returns:
    - str: The cleaned and normalized text.
    """
    text = text.lower()
    translator = str.maketrans("", "", string.punctuation)
    cleaned_text = text.translate(translator)
    return cleaned_text

cleaned_text = clean_and_normalize_text(text)
   

Following function computes the token frequencies and establishes the initial vocabulary:


In [4]:
def calculate_token_frequencies_and_vocabulary(text):
    """
    Calculate token frequencies and build a vocabulary from the input text.

    Parameters:
    - text (str): The input text.

    Returns:
    - tuple: A tuple containing a dictionary of token frequencies and a set of vocabulary.
    """
    # Create a dictionary to store token frequencies
    token_frequency = {} 
    vocabulary = set()
    ## START YOU CODE HERE
    words = text.split()
    for word in words:
        word += "_"
        characters = list(word)
        chars_with_spaces = ""
        word_len = len(word)
        counter = 0
        for character in characters:
            vocabulary.add(character)
            chars_with_spaces += character
            counter += 1
            if counter < word_len:
                chars_with_spaces += " "

        if chars_with_spaces in token_frequency:
            token_frequency[chars_with_spaces] += 1
        else:
            token_frequency[chars_with_spaces] = 1
    ## END
    return token_frequency, vocabulary

data, vocabulary = calculate_token_frequencies_and_vocabulary(cleaned_text)


Run the below cell to test your function:

In [5]:
test_data, test_vocabulary = calculate_token_frequencies_and_vocabulary("low low low low low lowest lowest newer newer newer newer newer newer wider wider wider new new")
assert test_data == {'l o w _': 5, 'l o w e s t _': 2, 'n e w e r _': 6, 'w i d e r _': 3, 'n e w _': 2}, "Test failed!"
assert test_vocabulary == {'_', 'd', 'e', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w'}, "Test failed!"
print("Success!")


Success!


Following function identifies the most frequent pair of adjacent symbols in the  data:


In [6]:
def calculate_most_frequent_symbol_pair(data):
    """
    Calculate most frequent pair of adjacent symbols in the given data.

    Parameters:
    - data (dict): A dictionary where keys are words and values are their frequencies.

    Returns:
    - tuple: A tuple containing the most frequent pair of symbols and its frequency.
    """   
    best_pair = ()
    frequency = 0
    ## START YOU CODE HERE
    pair_frequency = {}
    for word, freq in data.items():
        char_groups = word.split()
        group_len = len(char_groups)
        counter = 0
        while counter < group_len - 1:
            current = char_groups[counter]
            next = char_groups[counter+1]
            pair = (current, next)
            spaced_tuple = ' '.join(pair)
            if spaced_tuple in word:
                if pair in pair_frequency:
                    pair_frequency[pair] += freq
                else:
                    pair_frequency[pair] = freq
            counter += 1
    
    sorted_pairs_by_freq = sorted(pair_frequency.items(), key=lambda x:x[1], reverse=True)
    top_pair_freq = sorted_pairs_by_freq[0]
    best_pair = top_pair_freq[0]
    frequency = top_pair_freq[1]
    ## END    
    return best_pair, frequency


Run the below cell to test your function:

In [7]:
assert calculate_most_frequent_symbol_pair(test_data) == (('e', 'r'), 9), "Test failed!"
print("Success!")


Success!




This function merges a given symbol and update the data and vocabulary:


In [8]:
def merge_symbol(pair, data, vocabulary):
    """
    Merge the specified symbol pair into a single symbol and update the data and vocabulary.

    Parameters:
    - pair (tuple): The symbol pair to be merged.
    - data (dict): A dictionary where keys are words, and values are their frequencies.
    - vocabulary (set): A set containing the vocabulary of symbols.

    Returns:
    - tuple: A tuple containing the updated data, updated vocabulary, and the newly created symbol.
    """
    # Merge occurrences of the pair into a single symbol
    updated_data = {}
    ## START YOU CODE HERE
    spaced_tuple = ' '.join(pair)
    new_symbol = ''.join(pair)
    updated_vocabulary = vocabulary
    for word in data:
        pattern_1 = " "+spaced_tuple+" "
        pattern_2 = spaced_tuple+" "
        pattern_3 = " "+spaced_tuple
        exists = (pattern_1 in word) or (word.startswith(pattern_2)) or (word.endswith(pattern_3)) or (word == spaced_tuple)
        if exists:
            merged = word.replace(spaced_tuple, new_symbol)
            updated_data[merged] = data[word]
            updated_vocabulary.add(new_symbol)
        else:
            updated_data[word] = data[word]

    ## END   
    return updated_data, updated_vocabulary, new_symbol



We test our function:

In [9]:
assert merge_symbol(('e', 'r'), test_data, test_vocabulary) == ({'l o w _': 5, 'l o w e s t _': 2, 'n e w er _': 6, 
                                                                 'w i d er _': 3, 'n e w _': 2}, 
                                                                {'_', 'd', 'e', 'er', 'i', 'l', 'n', 'o', 'r', 's', 't', 'w'}, 
                                                                'er'), "Test failed!"

print("Success!")
assert merge_symbol(('e', 'r'), {'de ri_': 5, 'n e w e r _': 6}, 
                    {'_', 'd', 'e', 'i', 'n', 'r',}) == ({'de ri_': 5, 'n e w er _': 6}, 
                                                         {'_', 'd', 'e', 'er', 'i', 'n', 'r'}, 'er'), "Test failed!"
print("Success!")




Success!
Success!




We create the BPE function that will run a number of merges:


In [10]:
def bpe(data, vocabulary, num_merges):
    """
    Apply Byte Pair Encoding (BPE) to the given data for a specified number of merges.

    Parameters:
    - data (dict): A dictionary where keys are words and values are their frequencies.
    - vocabulary (set): A set containing the initial vocabulary of symbols.
    - num_merges (int): The number of merges to perform.

    Returns:
    - tuple: A tuple containing the updated data, updated vocabulary, and a list of merge rules (ranked).
    """
    merge_rules = []
    updated_data = data
    updated_vocabulary = vocabulary 
    ## START YOU CODE HERE
    for n in range(num_merges):
        best_pair, frequency = calculate_most_frequent_symbol_pair(updated_data)
        updated_data, updated_vocabulary, new_symbol = merge_symbol(best_pair, updated_data, updated_vocabulary)
    
        left = ' '.join(best_pair)
        merge_rules.append((left, new_symbol))
    ## END  
    return updated_data, updated_vocabulary, merge_rules


In [11]:
updated_test_data, updated_test_vocabulary, test_merge_rules = bpe(test_data, test_vocabulary, 8)
assert updated_test_data == {'low_': 5, 'low e s t _': 2, 'newer_': 6, 'w i d er_': 3, 'new _': 2}, "Test failed!"
assert updated_test_vocabulary == {'_','d','e','er','er_','i','l','lo','low','low_','n','ne','new','newer_','o','r','s','t','w'}, "Test failed!"
assert test_merge_rules == [('e r', 'er'), ('er _', 'er_'), ('n e', 'ne'), ('ne w', 'new'), 
                            ('l o', 'lo'), ('lo w', 'low'), ('new er_', 'newer_'), ('low _', 'low_')]
print("Success!")


Success!


# Training
Let's now learn our merge rules on the Hamlet text data.

In [12]:
updated_data, updated_vocabulary, merge_rules = bpe(data, vocabulary, 400)


# Tokenization

Let's construct our tokenizer, leveraging the merge rules acquired in the previous section.

The function below uses the merge_rules to tokenize a text.

In [13]:
def token_segmenter(text, merge_rules):
    """
    Tokenize the input text using BPE and the merge rules.

    Parameters:
    - text (str): The input text to be tokenized.
    - merge_rules (list): A list of merging rules obtained from the BPE algorithm.

    Returns:
    - list: A list of tokens obtained using token segmentation rules.
    """
    text = clean_and_normalize_text(text)
    ## START YOU CODE HERE

    tokenized_text = []
    
    token_frequency, vocabulary = calculate_token_frequencies_and_vocabulary(text)
    tokens = list(token_frequency.keys())

    for rule_pair in merge_rules:
        rule = rule_pair[0]
        merge = rule_pair[1]

        for idx, token in enumerate(tokens):
            new_token = token
            chars = new_token.split()
            if len(chars) == 1:
                continue

            i = 0
            word = []

            while i < len(chars) - 1:
                pair = chars[i] + " " + chars[i+1]
                if rule == pair:    
                    word.append(merge)
                    if i + 2 == len(chars) - 1:
                        word.append(chars[i+2])
                    
                    i += 2
                    continue
                    
                else:
                    word.append(chars[i])
                    if i == len(chars) - 2:
                        word.append(chars[i+1])
                    
                i += 1
                
            new_token = " ".join(word)
            tokens[idx] = new_token

    for token in tokens:
        tokenized_text.extend(token.split())
        
    ## END
    return tokenized_text
    

In [14]:
tokenized_text = token_segmenter("that which we call a rose by any other name would smell as sweet", merge_rules)
assert tokenized_text == ['that_', 'which_', 'we_', 'ca', 'll_', 'a_', 'ro', 
                          'se_', 'by_', 'an', 'y_', 'other_', 'n', 'a', 'me_', 
                          'would_', 's', 'm', 'ell_', 'as_', 'sw', 'eet_'], "Test failed!"
print("Success!")


Success!
